## Imports y Configuración Inicial

Librerías necesarias para el análisis de opciones y conexión con Interactive Brokers.



In [1]:
# !pip install ib_insync

from ib_insync import IB, util, Option, Stock, Contract
import asyncio
import nest_asyncio
import random
import time

from datetime import datetime
import pandas as pd

import sys
sys.path.insert(0, '../scripts')
from options_hedge import implied_vol_bisect, calculate_implied_q, StraddleForHedge, OptionsHedgeAnalyzer
from straddle import calculate_straddle_greeks_precise, calculate_straddle_greeks_american
from rates import get_risk_free_rate
from data_loader import load_treasury_curve

---

## 1. Conexión a TWS/IB Gateway

### Objetivo
Establecer una conexión robusta y confiable con TWS (Trader Workstation) o IB Gateway.

### ¿Qué hace esta celda?

1. **Importa librerías necesarias**
   - `ib_insync`: Interfaz Python para Interactive Brokers
   - `nest_asyncio`: Permite event loops anidados en Jupyter
   - `asyncio`: Programación asíncrona

2. **Configura la conexión**
   - HOST: `127.0.0.1` (localhost)
   - PORT: `7497` (Paper Trading) o `7496` (Live Trading)
   - CLIENT_ID: Generado aleatoriamente para evitar conflictos

3. **Función `connect_to_ib_async()`**
   - Verifica si ya existe una conexión activa
   - Reintentos automáticos (máx 3 intentos)
   - Manejo de conexiones corruptas
   - Timeouts configurables (20 segundos)
   - Mensajes de ayuda si falla la conexión

4. **Event Handlers**
   - Detecta desconexiones automáticamente
   - Muestra advertencias cuando se pierde la conexión

### Configuración de MarketDataType

Por defecto, se usa **MarketDataType 4** (Frozen/Delayed):
- GRATUITO - No requiere suscripción a datos en tiempo real
- Datos retrasados de 15 minutos
- Ideal para Paper Trading y desarrollo
- Disponible fuera del horario de mercado

### ¿Qué variables crea?

- `ib`: Objeto global de conexión (se usa en todas las celdas siguientes)

### Resultado esperado



In [ ]:
# Instalación de ib_insync si es necesario

# Permitir event loops anidados en Jupyter
nest_asyncio.apply()

# Configuración de conexión
HOST = "127.0.0.1"
PORT = 7497          # Paper trading (7497) o Live (7496)

# Crear instancia global de IB
ib = IB()

# Configurar handlers para manejar desconexiones
def on_disconnected():
    print("⚠ Desconectado de TWS/IB Gateway")

ib.disconnectedEvent += on_disconnected

async def connect_to_ib_async(max_retries=3):
    """
    Conecta a TWS/IB Gateway de forma robusta con reintentos
    
    Parámetros:
    - max_retries: Número máximo de intentos de conexión
    """
    
    for attempt in range(max_retries):
        try:
            # Si ya está conectado, verificar que funcione
            if ib.isConnected():
                try:
                    # Test simple para verificar que la conexión funciona
                    accounts = ib.managedAccounts()
                    print(f"✓ Ya conectado - clientId: {ib.client.clientId}")
                    print(f"  - Server version: {ib.client.serverVersion()}")
                    print(f"  - Managed accounts: {accounts}")
                    return True
                except Exception:
                    # La conexión está corrupta, desconectar
                    print("⚠ Conexión existente corrupta, desconectando...")
                    try:
                        ib.disconnect()
                    except:
                        pass
                    await asyncio.sleep(2)
            
            # Generar CLIENT_ID único basado en timestamp
            # Esto evita conflictos con conexiones previas
            client_id = random.randint(100, 999)
            
            print(f"\nIntento {attempt + 1}/{max_retries}: Conectando con clientId={client_id}...")
            
            # Intentar conectar con timeout más largo
            await ib.connectAsync(HOST, PORT, clientId=client_id, timeout=20, readonly=False)
            
            # Verificar que la conexión funciona
            await asyncio.sleep(1)
            accounts = ib.managedAccounts()
            
            print("✓ Conectado exitosamente")
            print(f"  - Client ID: {client_id}")
            print(f"  - Server version: {ib.client.serverVersion()}")
            print(f"  - Managed accounts: {accounts}")
            
            return True
            
        except Exception as e:
            error_msg = str(e)
            print(f"✗ Error en intento {attempt + 1}: {error_msg}")
            
            # Limpiar conexión fallida
            try:
                if ib.isConnected():
                    ib.disconnect()
            except:
                pass
            
            # Si es el último intento, mostrar mensaje de ayuda
            if attempt == max_retries - 1:
                print("\n" + "="*80)
                print("ERROR: No se pudo conectar a TWS/IB Gateway")
                print("="*80)
                print("\nVerifica lo siguiente:")
                print("  1. TWS o IB Gateway está ejecutándose")
                print(f"  2. Está configurado para aceptar conexiones API en el puerto {PORT}")
                print("  3. La configuración API permite conexiones desde 127.0.0.1")
                print("  4. No hay otras conexiones activas bloqueando el acceso")
                print("\nPasos para configurar TWS:")
                print("  - File -> Global Configuration -> API -> Settings")
                print("  - Marcar 'Enable ActiveX and Socket Clients'")
                print(f"  - Socket port: {PORT}")
                print("  - Trusted IP addresses: 127.0.0.1")
                print("="*80)
                return False
            
            # Esperar antes del siguiente intento
            wait_time = (attempt + 1) * 2
            print(f"  Esperando {wait_time} segundos antes del siguiente intento...")
            await asyncio.sleep(wait_time)
    
    return False

# Ejecutar la conexión
print("Iniciando conexión a TWS/IB Gateway...")
print("="*80)
connection_successful = await connect_to_ib_async(max_retries=3)

if connection_successful:
    print("\n✓ Listo para operar")
else:
    print("\n✗ No se pudo establecer conexión. Revisa los mensajes anteriores.")

Iniciando conexión a TWS/IB Gateway...

Intento 1/3: Conectando con clientId=178...
✓ Conectado exitosamente
  - Client ID: 178
  - Server version: 176
  - Managed accounts: ['DUP064233']

✓ Listo para operar


Error 10089, reqId 7: Los datos de mercado solicitados requieren suscripciones adicionales para API. Consulte el enlace en 'Conexiones de Datos de Mercado' para ver m\u00e1s detalles.SPY ARCA/TOP/ALL, contract: Stock(conId=756733, symbol='SPY', exchange='SMART', primaryExchange='ARCA', currency='USD', localSymbol='SPY', tradingClass='SPY')
Error 300, reqId 7: No se ha encontrado EId con el  tickerId:7
Error 10091, reqId 50: Parte de los datos de mercado solicitados requiere suscripciones adicionales para API. Consulte el enlace en 'Conexiones a datos de mercado' para ver m\u00e1s detalles.SPY ARCA/TOP/ALL, contract: Option(conId=843979094, symbol='SPY', lastTradeDateOrContractMonth='20260121', strike=685.0, right='C', multiplier='100', exchange='SMART', currency='USD', localSymbol='SPY   260121C00685000', tradingClass='SPY')
Error 10091, reqId 51: Parte de los datos de mercado solicitados requiere suscripciones adicionales para API. Consulte el enlace en 'Conexiones a datos de mercado'

---

## 2. Obtener Cadena de Opciones

### Objetivo
Descargar la cadena completa de opciones para SPY con un vencimiento específico.

### ¿Qué hace esta celda?

1. **Función `verify_connection()`**
  - Verifica que la conexión con IBKR esté activa
  - Lanza error si la conexión está corrupta o inactiva

2. **Función `get_option_chain_async()`**
  - **Parámetros:**
    - `symbol`: Símbolo del subyacente (default: "SPY")
    - `skip_expiries`: Número de vencimientos a saltar (1 = siguiente vencimiento, 2 = el siguiente, etc.)
  
  - **Proceso:**
    1. Cualifica el contrato del subyacente (SPY)
    2. Solicita parámetros de opciones disponibles
    3. Selecciona la cadena estándar (exchange=SMART)
    4. Filtra vencimientos futuros
    5. Selecciona vencimiento automáticamente
    6. Descarga todos los contratos (CALLs y PUTs)
    7. Construye DataFrame con la cadena completa

3. **Estructura de datos retornada**
  - Tabla pivoteada: cada fila = un strike
  - Columnas:
    - `strike`: Precio de ejercicio
    - `call_localSymbol`: Símbolo local del CALL
    - `call_conId`: ID del contrato CALL
    - `put_localSymbol`: Símbolo local del PUT
    - `put_conId`: ID del contrato PUT

### ¿Qué variables crea?

- `df_option_chain`: DataFrame con la cadena completa de opciones
- `expiry_date`: Fecha de vencimiento seleccionada (formato: YYYYMMDD)

### Parámetros configurables



In [3]:
async def verify_connection():
    """Verifica que la conexión esté activa y funcionando"""
    if not ib.isConnected():
        raise RuntimeError("No hay conexión activa. Ejecuta primero la celda de conexión.")
    
    try:
        # Test simple para verificar que la conexión funciona
        ib.managedAccounts()
    except Exception as e:
        raise RuntimeError(f"Conexión corrupta: {e}. Ejecuta nuevamente la celda de conexión.")

async def get_option_chain_async(symbol="SPY", skip_expiries=1):
    """
    Obtiene la cadena de opciones de forma robusta
    
    Parámetros:
    - symbol: Símbolo del subyacente (default: SPY)
    - skip_expiries: Número de vencimientos a saltar desde hoy (1 = siguiente vencimiento)
    
    Returns:
    - Tuple: (df_chain, expiry) - DataFrame con la cadena y fecha de vencimiento
    """
    
    print("="*80)
    print(f"OBTENIENDO CADENA DE OPCIONES: {symbol}")
    print("="*80 + "\n")
    
    # Verificar conexión
    await verify_connection()
    print(f"✓ Conexión activa (clientId={ib.client.clientId})\n")
    
    try:
        # -----------------------------
        # 1) Definir el subyacente
        # -----------------------------
        print(f"1. Cualificando contrato del subyacente {symbol}...")
        underlying = Stock(symbol, "SMART", "USD")
        await ib.qualifyContractsAsync(underlying)
        print(f"   ✓ {underlying.symbol} - ConId: {underlying.conId}\n")
        
        # -----------------------------
        # 2) Obtener parámetros de opciones
        # -----------------------------
        print("2. Solicitando parámetros de cadenas de opciones...")
        chains = await ib.reqSecDefOptParamsAsync(symbol, "", "STK", underlying.conId)
        
        if not chains:
            raise RuntimeError(f"No se encontraron cadenas de opciones para {symbol}")
        
        # Elegir cadena estándar (no FLEX)
        chain = next((c for c in chains if c.exchange == "SMART" and c.tradingClass == symbol), None)
        
        if not chain:
            raise RuntimeError(f"No se encontró cadena estándar para {symbol}")
        
        print(f"   ✓ Cadena: exchange={chain.exchange}, tradingClass={chain.tradingClass}, multiplier={chain.multiplier}\n")
        
        # -----------------------------
        # 3) Seleccionar vencimiento
        # -----------------------------
        expirations = sorted(chain.expirations)
        today = datetime.now().strftime("%Y%m%d")
        
        # Filtrar vencimientos futuros
        future_expiries = [e for e in expirations if e > today]
        
        if not future_expiries:
            raise RuntimeError("No hay vencimientos futuros disponibles")
        
        # Seleccionar vencimiento (skip_expiries para evitar 0DTE si es necesario)
        expiry_index = min(skip_expiries, len(future_expiries) - 1)
        expiry = future_expiries[expiry_index]
        
        print(f"3. Vencimiento seleccionado: {expiry}")
        print(f"   Vencimientos disponibles: {len(future_expiries)}")
        print(f"   Primeros 5: {future_expiries[:5]}\n")
        
        # -----------------------------
        # 4) Obtener contratos de opciones
        # -----------------------------
        print("4. Solicitando contratos de opciones a IBKR...")
        print("   (esto puede tomar unos segundos...)\n")
        
        tmpl = Contract(
            secType="OPT",
            symbol=symbol,
            currency="USD",
            exchange="SMART",
            lastTradeDateOrContractMonth=expiry,
            multiplier=str(chain.multiplier),
            tradingClass=chain.tradingClass,
        )
        
        details = await ib.reqContractDetailsAsync(tmpl)
        
        if not details:
            raise RuntimeError(
                f"No se recibieron contratos para {symbol} vencimiento {expiry}. "
                "Verifica permisos de market data o intenta otro vencimiento."
            )
        
        print(f"   ✓ Recibidos {len(details)} contratos\n")
        
        # -----------------------------
        # 5) Construir DataFrame
        # -----------------------------
        print("5. Procesando contratos...")
        
        rows = []
        for d in details:
            c = d.contract
            if c.symbol != symbol or c.secType != "OPT":
                continue
            rows.append({
                "expiry": c.lastTradeDateOrContractMonth,
                "right": c.right,
                "strike": c.strike,
                "conId": c.conId,
                "localSymbol": c.localSymbol,
                "exchange": c.exchange,
                "tradingClass": getattr(c, "tradingClass", None),
            })
        
        df = pd.DataFrame(rows)
        
        # Limpiar y filtrar
        df = df[df["expiry"] == expiry]
        df = df[df["right"].isin(["C", "P"])]
        df = df.dropna(subset=["strike", "conId"])
        df["strike"] = df["strike"].astype(float)
        
        print(f"   ✓ {len(df)} contratos procesados\n")
        
        # -----------------------------
        # 6) Crear tabla Option Chain
        # -----------------------------
        print("6. Creando tabla de option chain...")
        
        df_calls = (
            df[df["right"] == "C"][["strike", "localSymbol", "conId"]]
            .rename(columns={"localSymbol": "call_localSymbol", "conId": "call_conId"})
        )
        df_puts = (
            df[df["right"] == "P"][["strike", "localSymbol", "conId"]]
            .rename(columns={"localSymbol": "put_localSymbol", "conId": "put_conId"})
        )
        
        df_chain = (
            df_calls.merge(df_puts, on="strike", how="outer")
            .sort_values("strike")
            .reset_index(drop=True)
        )
        
        # Mostrar resumen
        print(f"\n{'='*80}")
        print(f"OPTION CHAIN {symbol} - Expiry: {expiry}")
        print(f"{'='*80}")
        print(f"Total strikes: {len(df_chain)}")
        print(f"Rango: ${df_chain['strike'].min():.0f} - ${df_chain['strike'].max():.0f}")
        print(f"\nPrimeros 10 strikes:")
        print(df_chain.head(10)[['strike', 'call_localSymbol', 'put_localSymbol']].to_string(index=False))
        print(f"{'='*80}\n")
        
        print("✓ Cadena de opciones obtenida exitosamente\n")
        
        return df_chain, expiry
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}\n")
        raise

# Ejecutar la función
try:
    df_option_chain, expiry_date = await get_option_chain_async("SPY", skip_expiries=5) # INIDCAR LOS VENCIMIENTOS A SALTAR DESDE HOY
    print(f"Variable 'df_option_chain' creada con {len(df_option_chain)} strikes")
    print(f"Variable 'expiry_date' = {expiry_date}")
except Exception as e:
    print(f"No se pudo obtener la cadena de opciones: {e}")

OBTENIENDO CADENA DE OPCIONES: SPY

✓ Conexión activa (clientId=178)

1. Cualificando contrato del subyacente SPY...
   ✓ SPY - ConId: 756733

2. Solicitando parámetros de cadenas de opciones...
   ✓ Cadena: exchange=SMART, tradingClass=SPY, multiplier=100

3. Vencimiento seleccionado: 20260121
   Vencimientos disponibles: 32
   Primeros 5: ['20260113', '20260114', '20260115', '20260116', '20260120']

4. Solicitando contratos de opciones a IBKR...
   (esto puede tomar unos segundos...)

   ✓ Recibidos 374 contratos

5. Procesando contratos...
   ✓ 374 contratos procesados

6. Creando tabla de option chain...

OPTION CHAIN SPY - Expiry: 20260121
Total strikes: 187
Rango: $500 - $950

Primeros 10 strikes:
 strike      call_localSymbol       put_localSymbol
  500.0 SPY   260121C00500000 SPY   260121P00500000
  505.0 SPY   260121C00505000 SPY   260121P00505000
  510.0 SPY   260121C00510000 SPY   260121P00510000
  515.0 SPY   260121C00515000 SPY   260121P00515000
  520.0 SPY   260121C005200

---

## 3. Obtener Precio Actual de SPY

###  Objetivo
Obtener el precio actual del subyacente (SPY) para identificar el strike ATM (At The Money).

###  ¿Qué hace esta celda?

Esta celda implementa un **sistema de fallback robusto en 3 niveles** para asegurar que siempre obtengamos un precio de referencia:

#### **MÉTODO 1: IBKR (Prioridad Alta)**
- Usa la conexión activa con IBKR
- Solicita datos de mercado retrasados (MarketDataType 4)
- Prioridad de precios:
  1. `Last` (último precio negociado)
  2. `Close` (precio de cierre)
  3. `Bid` (mejor oferta de compra)
-  **Gratis** - No requiere suscripción a datos en tiempo real

#### **MÉTODO 2: Yahoo Finance (Fallback)**
- Si IBKR no devuelve precio válido
- Descarga datos usando la librería `yfinance`
- Instala automáticamente `yfinance` si no está disponible
- Prioridad de precios:
  1. `regularMarketPrice` (precio de mercado regular)
  2. `currentPrice` (precio actual)
  3. `previousClose` (cierre anterior)
  4. `Close` del historial
-  **Gratis** - No requiere API key

#### **MÉTODO 3: Precio Manual (Última Opción)**
- Si ambos métodos anteriores fallan
- Usa precio de seguridad hardcodeado (~$595.00)
-  **Advertencia clara al usuario** para verificar conexión
- Permite que el notebook continúe funcionando

###  ¿Qué variables crea?

- `spy_price`: Precio actual de SPY (float)

###  Resultado esperado

**Caso exitoso (IBKR):**
```
 ÉXITO - Precio obtenido de IBKR
================================================================================
PRECIO SPY: $689.51
Fuente: IBKR (Last)
================================================================================
```

**Caso exitoso (Yahoo):**
```
 ÉXITO - Precio obtenido de Yahoo Finance
================================================================================
PRECIO SPY: $689.51
Fuente: Yahoo Finance (Regular Market Price)
================================================================================
```

**Caso fallback:**
```
 ADVERTENCIA: Usando precio manual de seguridad
================================================================================
PRECIO SPY: $595.00
Fuente: Precio Manual de Seguridad
================================================================================

  IMPORTANTE: Este es un precio de respaldo.
   Verifica la conexión IBKR o la disponibilidad de Yahoo Finance
```

###  Tiempo de ejecución

- IBKR: **1-3 segundos**
- Yahoo Finance: **2-5 segundos**
- Manual: **Instantáneo**

###  Notas importantes

-  **Triple capa de seguridad** - Siempre obtendrás un precio
-  **No bloquea el flujo** - El notebook puede continuar
-  **Instalación automática** de dependencias (yfinance)
-  **Mensajes claros** indicando la fuente del precio
-  Si ves el precio manual, verifica tu conexión

###  ¿Cuándo actualizar el precio manual?

Si el precio de SPY cambia significativamente (~>10%), actualiza esta línea en el código:

```python
safety_price = 595.00  # <-- Actualizar aquí
```

---

** Ejecuta esta celda después de obtener la cadena de opciones**

In [4]:
async def get_spy_price_async():
    """
    Obtiene el precio actual de SPY con fallback robusto:
    1. Intenta obtenerlo de IBKR (datos delayed/frozen)
    2. Si falla, descarga de Yahoo Finance
    3. Si falla, usa precio manual de seguridad
    
    Returns:
    - float: Precio actual de SPY
    """
    
    print("="*80)
    print("OBTENIENDO PRECIO DE SPY (Con Fallback Robusto)")
    print("="*80 + "\n")
    
    # =========================================================================
    # MÉTODO 1: Obtener de IBKR
    # =========================================================================
    print(" MÉTODO 1: Intentando obtener precio desde IBKR...")
    print("-" * 80)
    
    if ib.isConnected():
        try:
            # Configurar datos retrasados/congelados (MarketDataType 4)
            ib.reqMarketDataType(4)
            print("✓ Conexión activa - MarketDataType(4) configurado\n")
            
            # Crear contrato de SPY
            print("1. Cualificando contrato SPY...")
            spy = Stock("SPY", "SMART", "USD")
            await ib.qualifyContractsAsync(spy)
            print(f"   ✓ Contrato cualificado - ConId: {spy.conId}\n")
            
            # Solicitar datos de mercado (snapshot)
            print("2. Solicitando precio de mercado...")
            spy_ticker = ib.reqMktData(spy, "", snapshot=True, regulatorySnapshot=False)
            
            # Esperar a recibir datos (máximo 3 segundos)
            spy_price = None
            for i in range(15):
                await asyncio.sleep(0.2)
                
                # Intentar obtener precio en orden de prioridad
                if spy_ticker.last and spy_ticker.last > 0:
                    spy_price = spy_ticker.last
                    price_type = "Last"
                    break
                elif spy_ticker.close and spy_ticker.close > 0:
                    spy_price = spy_ticker.close
                    price_type = "Close"
                    break
                elif spy_ticker.bid and spy_ticker.bid > 0:
                    spy_price = spy_ticker.bid
                    price_type = "Bid"
                    break
            
            # Cancelar suscripción
            ib.cancelMktData(spy)
            
            # Verificar resultado
            if spy_price and spy_price > 0:
                print(f" ÉXITO - Precio obtenido de IBKR")
                print("="*80)
                print(f"PRECIO SPY: ${spy_price:.2f}")
                print(f"Fuente: IBKR ({price_type})")
                print("="*80 + "\n")
                return spy_price
            else:
                print(" IBKR no devolvió precio válido\n")
                
        except Exception as e:
            print(f" Error al obtener precio de IBKR: {e}\n")
    else:
        print(" No hay conexión activa con IBKR\n")
    
    # =========================================================================
    # MÉTODO 2: Obtener de Yahoo Finance
    # =========================================================================
    print(" MÉTODO 2: Intentando obtener precio desde Yahoo Finance...")
    print("-" * 80)
    
    try:
        # Intentar importar yfinance
        try:
            import yfinance as yf
        except ImportError:
            print(" Librería 'yfinance' no instalada.")
            print("  Instalando automáticamente...")
            import subprocess
            import sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
            import yfinance as yf
            print("✓ yfinance instalado correctamente\n")
        
        print("1. Descargando datos de Yahoo Finance...")
        spy_ticker = yf.Ticker("SPY")
        
        # Obtener datos de diferentes fuentes
        spy_info = spy_ticker.info
        spy_history = spy_ticker.history(period="1d")
        
        spy_price = None
        price_type = None
        
        # Prioridad: regularMarketPrice > currentPrice > previousClose > Close del history
        if 'regularMarketPrice' in spy_info and spy_info['regularMarketPrice']:
            spy_price = spy_info['regularMarketPrice']
            price_type = "Regular Market Price"
        elif 'currentPrice' in spy_info and spy_info['currentPrice']:
            spy_price = spy_info['currentPrice']
            price_type = "Current Price"
        elif 'previousClose' in spy_info and spy_info['previousClose']:
            spy_price = spy_info['previousClose']
            price_type = "Previous Close"
        elif not spy_history.empty:
            spy_price = spy_history['Close'].iloc[-1]
            price_type = "Last Close (History)"
        
        if spy_price and spy_price > 0:
            print(f" ÉXITO - Precio obtenido de Yahoo Finance")
            print("="*80)
            print(f"PRECIO SPY: ${spy_price:.2f}")
            print(f"Fuente: Yahoo Finance ({price_type})")
            print("="*80 + "\n")
            return float(spy_price)
        else:
            print(" Yahoo Finance no devolvió precio válido\n")
            
    except Exception as e:
        print(f" Error al obtener precio de Yahoo Finance: {e}\n")
    
    # =========================================================================
    # MÉTODO 3: Precio Manual de Seguridad
    # =========================================================================
    print(" MÉTODO 3: Usando precio manual de seguridad...")
    print("-" * 80)
    
    # Precio de seguridad (actualizar manualmente si es necesario)
    # Este es un precio aproximado del SPY a finales de diciembre 2024
    safety_price = 595.00
    
    print(f" ADVERTENCIA: Usando precio manual de seguridad")
    print("="*80)
    print(f"PRECIO SPY: ${safety_price:.2f}")
    print(f"Fuente: Precio Manual de Seguridad")
    print("="*80)
    print("\n  IMPORTANTE: Este es un precio de respaldo.")
    print("   Verifica la conexión IBKR o la disponibilidad de Yahoo Finance")
    print("   para obtener precios actualizados en tiempo real.\n")
    
    return safety_price

# Ejecutar la función
spy_price = await get_spy_price_async()

print(f"✓ Variable 'spy_price' = ${spy_price:.2f}")

OBTENIENDO PRECIO DE SPY (Con Fallback Robusto)

 MÉTODO 1: Intentando obtener precio desde IBKR...
--------------------------------------------------------------------------------
✓ Conexión activa - MarketDataType(4) configurado

1. Cualificando contrato SPY...
   ✓ Contrato cualificado - ConId: 756733

2. Solicitando precio de mercado...
 IBKR no devolvió precio válido

 MÉTODO 2: Intentando obtener precio desde Yahoo Finance...
--------------------------------------------------------------------------------
1. Descargando datos de Yahoo Finance...
 ÉXITO - Precio obtenido de Yahoo Finance
PRECIO SPY: $695.16
Fuente: Yahoo Finance (Regular Market Price)

✓ Variable 'spy_price' = $695.16


---

## 4. Obtener Precios Bid/Ask de Opciones

###  Objetivo
Obtener precios de mercado (bid/ask) para las opciones cercanas al precio ATM de SPY.

###  ¿Qué hace esta celda?

#### **Función `get_option_prices_async()`**

**Parámetros:**
- `df_chain`: DataFrame con la cadena de opciones (del paso anterior)
- `expiry_date`: Fecha de vencimiento (del paso anterior)
- `center_strike`: Strike central específico (opcional)
- `max_strikes`: Número máximo de strikes a procesar (default: 20)
- `reference_price`: Precio de referencia de SPY (del paso anterior)

**Proceso:**

| Paso | Descripción | Detalles |
|------|-------------|----------|
| **1. Configuración de datos** | Usa MarketDataType(4) - Frozen/Delayed |  **GRATUITO** - No requiere suscripción a datos en tiempo real. Ideal para Paper Trading y desarrollo |
| **2. Selección de strikes** | Filtra strikes cercanos al ATM | Usa el precio de referencia de SPY. Encuentra el strike más cercano (ATM). Selecciona `max_strikes` alrededor del ATM. Ejemplo: Si ATM=$690 y max_strikes=20 → strikes de $680 a $700 |
| **3. Creación de contratos** | Prepara objetos Option | Crea objetos `Option` para cada CALL y PUT. Cualifica contratos con IBKR. Prepara para solicitud masiva de datos |
| **4. Solicitud de precios** | **MODO SNAPSHOT** -  Solicitud paralela masiva |  **Solicitud paralela masiva** - Todos los contratos a la vez. `snapshot=True` - Solicita dato puntual (no streaming). Espera inteligente: 2-4 segundos con verificación del 90%.  **Mucho más rápido** que solicitudes secuenciales |
| **5. Procesamiento** | Extrae y organiza datos | Extrae: bid, ask, last, volume. Prioriza: Last → Close (para datos delayed). Guarda en DataFrame con columnas separadas para CALL y PUT. Cancela suscripciones para liberar recursos |
| **6. Visualización** | Muestra resultados formateados | Tabla formateada con todos los strikes. Precios bid/ask/last para CALLs y PUTs. Estadísticas de cobertura |

###  ¿Qué variables crea?

- `df_with_prices`: DataFrame con strikes y precios completos

**Columnas del DataFrame:**
- `strike`: Precio de ejercicio
- `call_localSymbol`: Símbolo local del CALL
- `call_conId`: ID del contrato CALL
- `call_bid`: Precio bid del CALL
- `call_ask`: Precio ask del CALL
- `call_last`: Último precio del CALL
- `call_volume`: Volumen del CALL
- `put_localSymbol`: Símbolo local del PUT
- `put_conId`: ID del contrato PUT
- `put_bid`: Precio bid del PUT
- `put_ask`: Precio ask del PUT
- `put_last`: Último precio del PUT
- `put_volume`: Volumen del PUT

###  Resultado esperado

```
====================================================================================================
PRECIOS DE OPCIONES SPY - Expiry: 20251231 (Snapshot Mode)
====================================================================================================

 strike  call_bid  call_ask  call_last  put_bid  put_ask  put_last
  680.0     10.20     11.23      11.02     0.51     0.52      0.52
  681.0      9.60     10.00       9.80     0.58     0.59      0.57
  ...
  690.0      2.59      2.60       2.59     2.28     2.30      2.28  ← ATM
  ...
  700.0      0.09      0.10       0.09     8.84    10.20      8.56

====================================================================================================
Strikes: 21 | Contratos con precios: 42/42
====================================================================================================
```

###  Ventajas del Modo Snapshot

-  **Solicitud paralela masiva** - Todos los contratos simultáneamente
-  **No requiere streaming** - Solo un dato puntual
-  **Gratis con datos delayed** - No requiere suscripción
-  **Espera inteligente** - Verifica que llegue el 90% de datos
-  **Limpieza automática** - Cancela suscripciones al terminar

###  Parámetros configurables

```python
# Obtener más strikes (ejemplo: 40 strikes)
df_with_prices = await get_option_prices_async(
    df_option_chain, 
    expiry_date, 
    max_strikes=40,
    reference_price=spy_price
)

# Forzar un strike central específico
df_with_prices = await get_option_prices_async(
    df_option_chain, 
    expiry_date, 
    center_strike=690.0,
    reference_price=spy_price
)
```

###  Uso para Long Straddle

Con estos datos puedes:
1. Identificar el strike ATM exacto
2. Ver el costo del CALL ATM (call_ask)
3. Ver el costo del PUT ATM (put_ask)
4. **Costo total = call_ask + put_ask**
5. Calcular puntos de equilibrio (strike ± costo total)

###  Notas importantes

-  Usa precios **delayed/frozen** - GRATUITOS
-  Modo snapshot - Mucho más rápido y eficiente
-  Filtra strikes cercanos al ATM automáticamente
-  Durante horario de mercado: datos con 15 min de retraso
-  Fuera de horario: datos "frozen" del último cierre

---

** Ejecuta esta celda después de obtener el precio de SPY**

In [5]:
async def get_option_prices_async(df_chain, expiry_date, center_strike=None, max_strikes=20, reference_price=None):
    """
    Obtiene precios bid/ask de forma robusta usando Snapshot y MarketDataType 4 (Frozen).
    
    Parámetros:
    - df_chain: DataFrame con la cadena de opciones
    - expiry_date: Fecha de vencimiento
    - center_strike: Strike central específico (opcional)
    - max_strikes: Número máximo de strikes a procesar
    - reference_price: Precio de referencia del subyacente (si ya se obtuvo previamente)
    """
    
    print("="*80)
    print(f"OBTENIENDO PRECIOS DE OPCIONES - Expiry: {expiry_date}")
    print("="*80 + "\n")
    
    # Configurar datos congelados/retrasados (Frozen es mejor para pruebas off-hours o delayed)
    ib.reqMarketDataType(4) 
    print("✓ Configurado: MarketDataType(4) - Frozen/Delayed (Más robusto para Paper Trading)\n")
    
    try:
        # -----------------------------
        # 1) Obtener precio de referencia (SPY)
        # -----------------------------
        spy_price = reference_price  # Usar el precio proporcionado como parámetro
        
        if spy_price:
            print(f"1. Usando precio de referencia proporcionado: ${spy_price:.2f}")
            print("   (Obtenido previamente de Yahoo Finance o IBKR)\n")
        else:
            print("1. Obteniendo precio de referencia del subyacente (SPY)...")
            
            try:
                spy = Stock("SPY", "SMART", "USD")
                await ib.qualifyContractsAsync(spy)
                
                # Usamos snapshot=True para asegurar un dato puntual
                spy_ticker = ib.reqMktData(spy, "", True, False)
                
                # Esperar hasta obtener un dato válido (máx 2 seg)
                for _ in range(10):
                    await asyncio.sleep(0.2)
                    # Prioridad: Last -> Close -> MarketPrice
                    if spy_ticker.last and spy_ticker.last > 0:
                        spy_price = spy_ticker.last
                        break
                    elif spy_ticker.close and spy_ticker.close > 0:
                        spy_price = spy_ticker.close
                        break
                
                # Si sigue fallando, intentar marketPrice() genérico de ib_insync
                if not spy_price:
                    spy_price = spy_ticker.marketPrice()
                    
                if spy_price and spy_price > 0:
                    print(f"   ✓ Precio SPY detectado: ${spy_price:.2f}\n")
                else:
                    print("    ADVERTENCIA: No se pudo obtener precio SPY. Usando strike central por defecto.\n")
                    
            except Exception as e:
                print(f"   Error obteniendo SPY: {e}\n")
        
        # -----------------------------
        # 2) Seleccionar strikes
        # -----------------------------
        print("2. Seleccionando strikes...")
        
        # Determinar strike central
        if center_strike is None:
            if spy_price and spy_price > 0:
                temp_df = df_chain.copy()
                temp_df['distance'] = abs(temp_df['strike'] - spy_price)
                center_strike = temp_df.loc[temp_df['distance'].idxmin(), 'strike']
            else:
                # Fallback si no hay precio de SPY: usar la mediana de la cadena
                center_strike = df_chain['strike'].median()
                print(f"   (Usando mediana de la cadena como referencia: {center_strike})")
        
        # Filtrar strikes alrededor del centro
        strike_range = max_strikes // 2
        df_filtered = df_chain.copy().sort_values('strike').reset_index(drop=True)
        
        df_filtered['distance'] = abs(df_filtered['strike'] - center_strike)
        center_idx = df_filtered['distance'].idxmin()
        
        start_idx = max(0, center_idx - strike_range)
        end_idx = min(len(df_filtered), center_idx + strike_range + 1)
        df_filtered = df_filtered.iloc[start_idx:end_idx].copy()
        df_filtered = df_filtered.drop('distance', axis=1).reset_index(drop=True)
        
        print(f"   ✓ Strike central: ${center_strike:.2f}")
        print(f"   ✓ Rango: ${df_filtered['strike'].min():.0f} - ${df_filtered['strike'].max():.0f}")
        print(f"   ✓ {len(df_filtered)} strikes seleccionados\n")
        
        # -----------------------------
        # 3) Crear y Cualificar contratos
        # -----------------------------
        print("3. Creando y cualificando contratos...")
        
        contracts = []
        contract_info = []  # Lista en lugar de diccionario (evita problemas de hash)
        
        for idx, row in df_filtered.iterrows():
            strike = row['strike']
            
            # CALL
            if pd.notna(row.get('call_conId')):
                call = Option('SPY', expiry_date, strike, 'C', 'SMART', tradingClass='SPY')
                contracts.append(call)
                contract_info.append(('call', idx))  # Guardar tipo y índice
            
            # PUT
            if pd.notna(row.get('put_conId')):
                put = Option('SPY', expiry_date, strike, 'P', 'SMART', tradingClass='SPY')
                contracts.append(put)
                contract_info.append(('put', idx))  # Guardar tipo y índice
        
        qualified = await ib.qualifyContractsAsync(*contracts)
        print(f"   ✓ {len(qualified)} contratos listos para solicitar datos\n")
        
        # -----------------------------
        # 4) Obtener precios (SNAPSHOT / PARALELO)
        # -----------------------------
        print("4. Solicitando Snapshots de mercado (Paralelo)...")
        
        # Inicializar columnas en el DF
        cols_to_init = ['call_bid', 'call_ask', 'call_last', 'call_volume', 
                       'put_bid', 'put_ask', 'put_last', 'put_volume']
        for col in cols_to_init:
            df_filtered[col] = None

        # SOLICITUD MASIVA (Mucho más rápido)
        # Usamos snapshot=True. Esto es clave para cuentas sin datos live.
        tickers_list = []
        for contract in qualified:
            # snapshot=True, regulatorySnapshot=False
            t = ib.reqMktData(contract, "", True, False)
            tickers_list.append(t)
        
        print(f"    Esperando datos (2-4 segundos)...")
        # Esperamos un tiempo prudencial para que lleguen los snapshots
        await asyncio.sleep(20) 
        
        # Opcional: Espera activa inteligente (esperar a que la mayoría tenga datos)
        for _ in range(20):  # Máx 4 segundos extra
            filled = sum(1 for t in tickers_list if t.last or t.bid or t.ask or t.close)
            if filled >= len(qualified) * 0.9:  # Si tenemos el 90% de los datos
                break
            await asyncio.sleep(0.2)

        # -----------------------------
        # 5) Procesar resultados
        # -----------------------------
        valid_count = 0

        # Procesamos los resultados usando índices
        # contract_info, qualified y tickers_list están alineados por índice
        for i, (qualified_contract, ticker) in enumerate(zip(qualified, tickers_list)):

            # Obtener información usando el índice
            if i >= len(contract_info):
                continue
                
            opt_type, df_idx = contract_info[i]

            # Lógica de precios (Priorizamos Last, luego Close para Delayed)
            bid = ticker.bid if (ticker.bid and ticker.bid > 0) else None
            ask = ticker.ask if (ticker.ask and ticker.ask > 0) else None
            last = ticker.last if (ticker.last and ticker.last > 0) else ticker.close
            volume = ticker.volume

            # Asignar al DataFrame
            prefix = 'call' if opt_type == 'call' else 'put'

            df_filtered.at[df_idx, f'{prefix}_bid'] = bid
            df_filtered.at[df_idx, f'{prefix}_ask'] = ask
            df_filtered.at[df_idx, f'{prefix}_last'] = last
            df_filtered.at[df_idx, f'{prefix}_volume'] = volume

            if bid or ask or last:
                valid_count += 1

            # Limpiar suscripción (buena práctica)
            ib.cancelMktData(qualified_contract)

        # -----------------------------
        # 6) Mostrar resultados
        # -----------------------------
        print(f"\n{'='*100}")
        print(f"PRECIOS DE OPCIONES SPY - Expiry: {expiry_date} (Snapshot Mode)")
        print(f"{'='*100}\n")
        
        display_cols = ['strike', 'call_bid', 'call_ask', 'call_last', 'put_bid', 'put_ask', 'put_last']
        
        # Formatear para imprimir bonito (rellenar NaN con guiones)
        df_print = df_filtered[display_cols].fillna('-')
        print(df_print.to_string(index=False))
        
        print(f"\n{'='*100}")
        print(f"Strikes: {len(df_filtered)} | Contratos con precios: {valid_count}/{len(qualified)}")
        print(f"{'='*100}\n")
        
        return df_filtered
        
    except Exception as e:
        print(f"\n✗ ERROR CRÍTICO: {e}")
        import traceback
        traceback.print_exc()
        return df_chain  # Devolver original en caso de error

# ---------------------------------------------------------
# Bloque de ejecución principal
# ---------------------------------------------------------
# Ejecutar la función pasando el precio obtenido de Yahoo Finance
df_with_prices = await get_option_prices_async(df_option_chain, expiry_date, reference_price=spy_price)

OBTENIENDO PRECIOS DE OPCIONES - Expiry: 20260121

✓ Configurado: MarketDataType(4) - Frozen/Delayed (Más robusto para Paper Trading)

1. Usando precio de referencia proporcionado: $695.16
   (Obtenido previamente de Yahoo Finance o IBKR)

2. Seleccionando strikes...
   ✓ Strike central: $695.00
   ✓ Rango: $685 - $705
   ✓ 21 strikes seleccionados

3. Creando y cualificando contratos...
   ✓ 42 contratos listos para solicitar datos

4. Solicitando Snapshots de mercado (Paralelo)...
    Esperando datos (2-4 segundos)...

PRECIOS DE OPCIONES SPY - Expiry: 20260121 (Snapshot Mode)

 strike  call_bid  call_ask  call_last  put_bid  put_ask  put_last
  685.0     12.13     12.42      11.65     1.37     1.39      1.40
  686.0     11.26     11.56      11.17     1.52     1.54      1.53
  687.0     10.43     10.72       9.26     1.68     1.70      1.62
  688.0      9.61      9.92       9.33     1.86     1.89      1.89
  689.0      8.83      9.13       9.20     2.07     2.09      2.10
  690.0    

C:\Users\diego\AppData\Local\Temp\ipykernel_30984\596891257.py:196: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_print = df_filtered[display_cols].fillna('-')


## 5. Cálculo de Griegas del Straddle

Este proceso determina la volatilidad implícita (IV) a partir de los precios de mercado para capturar el skew real. El objetivo es obtener una medición precisa de las griegas del straddle basándose en las condiciones actuales.

* Metodologías de Volatilidad*

Existen dos enfoques principales para la gestión de la volatilidad en el cálculo de griegas:

1. **Retrotest Histórico (Proxy VIX)** Práctica llevada a cabo en los apartados anteriores.
    * **Método**: Utiliza el VIX/100 como proxy de volatilidad único.
    * **Justificación**: Facilita el análisis histórico donde no siempre se dispone de cadenas de opciones completas.
    * **Limitación**: No captura el skew específico del ATM de SPY.

2. **Análisis en Tiempo Real (Datos de Mercado)**
    * **Método**: Inversión del modelo BSM para hallar la IV de la Call y la Put por separado. Se utiliza el promedio de ambas para las griegas del straddle.
    * **Ventaja**: Refleja con precisión las condiciones actuales del mercado.

---

**Nota sobre Precisión**: 
Para el análisis de straddles At-The-Money (ATM), el uso de la volatilidad promedio entre ambos contratos proporciona una aproximación adecuada, dado que las IVs de Calls y Puts ATM suelen ser similares.


In [6]:
# Parámetros del mercado
S = spy_price  # Precio del subyacente

# ==============================================================================
# Calibración de parámetros: Tasa libre de riesgo y Dividend Yield
# ==============================================================================

print("="*80)
print("CALIBRACIÓN DE PARÁMETROS DE MERCADO")
print("="*80 + "\n")

# ------------------------------------------------------------------------------
# 1. Tasa libre de riesgo - Interpolación basada en curva del Tesoro
# ------------------------------------------------------------------------------
print("1. Cargando curva del Tesoro desde FRED...")

from datetime import timedelta

# Cargar curva del Tesoro de los últimos 60 días
# FRED solo provee datos históricos, NO futuros
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=60)).strftime("%Y-%m-%d")

treasury_data = load_treasury_curve(start_date, end_date)

# Validar que obtuvimos datos
if treasury_data.empty:
    raise ValueError(
        "No se pudo obtener datos de la curva del Tesoro desde FRED. "
        "Verifica tu FRED_API_KEY en el archivo .env"
    )

# Usar la curva más reciente disponible (último día hábil)
latest_date = treasury_data.index[-1]
curve_row = treasury_data.loc[latest_date]

print(f"   ✓ Curva del Tesoro utilizada: {latest_date}")
print(f"     (Datos más recientes disponibles desde FRED)")
print(f"\n   Tasas del Tesoro:")
print(f"     1 Mes  (30d):  {curve_row[30]:.4f} ({curve_row[30]*100:.2f}%)")
print(f"     3 Meses (90d):  {curve_row[90]:.4f} ({curve_row[90]*100:.2f}%)")
print(f"     6 Meses (180d): {curve_row[180]:.4f} ({curve_row[180]*100:.2f}%)")
print(f"     1 Año (365d):   {curve_row[365]:.4f} ({curve_row[365]*100:.2f}%)")
print()

# Calcular tenor en días hasta vencimiento (desde HOY)
expiry = datetime.strptime(expiry_date, "%Y%m%d")
tenor_days = (expiry - datetime.now()).days

# Interpolar tasa libre de riesgo para el tenor específico
r = get_risk_free_rate(tenor_days, curve_row)

print(f"2. Tasa libre de riesgo para el vencimiento:")
print(f"   Expiry:          {expiry_date}")
print(f"   Tenor:           {tenor_days} días")
print(f"   r (interpolada): {r:.4f} ({r*100:.2f}%)")
print()

# ------------------------------------------------------------------------------
# 2. Calcular tiempo hasta vencimiento (en años)
# ------------------------------------------------------------------------------
today = datetime.now()
T = (expiry - today).days / 365.0

print(f"3. Tiempo hasta vencimiento:")
print(f"   T = {T:.4f} años ({(expiry - today).days} días)")
print()

# ------------------------------------------------------------------------------
# 3. Obtener strike ATM (el más cercano al precio actual)
# ------------------------------------------------------------------------------
df_with_prices['distance'] = abs(df_with_prices['strike'] - S)
atm_row = df_with_prices.loc[df_with_prices['distance'].idxmin()]
K = atm_row['strike']

print(f"4. Strike ATM seleccionado:")
print(f"   Precio SPY:  ${S:.2f}")
print(f"   Strike ATM:  ${K:.0f}")
print()

# ------------------------------------------------------------------------------
# 4. Calcular mid price para call y put ATM
# ------------------------------------------------------------------------------
call_mid = (atm_row['call_bid'] + atm_row['call_ask']) / 2
put_mid = (atm_row['put_bid'] + atm_row['put_ask']) / 2

print(f"5. Precios de mercado ATM:")
print(f"   CALL mid: ${call_mid:.2f}")
print(f"   PUT mid:  ${put_mid:.2f}")
print()

# ------------------------------------------------------------------------------
# 5. Calcular dividend yield implícito desde put-call parity
# ------------------------------------------------------------------------------
q = calculate_implied_q(S, K, T, r, call_mid, put_mid)

print(f"6. Dividend yield (implied from market):")
print(f"   q = {q:.4f} ({q*100:.2f}%)")

print("\n" + "="*80)
print("✓ CALIBRACIÓN COMPLETADA")
print("="*80 + "\n")

CALIBRACIÓN DE PARÁMETROS DE MERCADO

1. Cargando curva del Tesoro desde FRED...
   ✓ Curva del Tesoro utilizada: 2026-01-09
     (Datos más recientes disponibles desde FRED)

   Tasas del Tesoro:
     1 Mes  (30d):  0.0370 (3.70%)
     3 Meses (90d):  0.0362 (3.62%)
     6 Meses (180d): 0.0357 (3.57%)
     1 Año (365d):   0.0352 (3.52%)

2. Tasa libre de riesgo para el vencimiento:
   Expiry:          20260121
   Tenor:           8 días
   r (interpolada): 0.0370 (3.70%)

3. Tiempo hasta vencimiento:
   T = 0.0219 años (8 días)

4. Strike ATM seleccionado:
   Precio SPY:  $695.16
   Strike ATM:  $695

5. Precios de mercado ATM:
   CALL mid: $4.75
   PUT mid:  $3.90

6. Dividend yield (implied from market):
   q = -0.0083 (-0.83%)

✓ CALIBRACIÓN COMPLETADA



In [7]:
# Calcular volatilidad implicita
iv_call = implied_vol_bisect(call_mid, S, K, T, r, q, "call")
iv_put = implied_vol_bisect(put_mid, S, K, T, r, q, "put")

print(f"\nVolatilidad implicita CALL ATM: {iv_call:.4f} ({iv_call*100:.2f}%)")
print(f"Volatilidad implicita PUT ATM:  {iv_put:.4f} ({iv_put*100:.2f}%)")


Volatilidad implicita CALL ATM: 0.1049 (10.49%)
Volatilidad implicita PUT ATM:  0.1049 (10.49%)


In [8]:
# Calcular griegas del straddle
greeks = calculate_straddle_greeks_precise(S, K, T, r, q, iv_call, iv_put)

print("="*50)
print(f"GRIEGAS DEL STRADDLE ATM (K=${K:.0f})")
print("="*50)
print(f"Delta:  {greeks['delta']:+.4f}")
print(f"Gamma:  {greeks['gamma']:.4f}")
print(f"Vega:   {greeks['vega']:.4f}")
print(f"Theta:  {greeks['theta']:.4f} (por dia)")
print(f"Rho:    {greeks['rho']:.4f}")

GRIEGAS DEL STRADDLE ATM (K=$695)
Delta:  +0.0689
Gamma:  0.0736
Vega:   0.8182
Theta:  -0.5417 (por dia)
Rho:    0.0086


### 5.1 Comparación: Griegas BSM vs Americanas (QuantLib)

Para opciones americanas sobre SPY, las griegas pueden diferir ligeramente de las europeas (BSM) debido al valor del ejercicio anticipado. Esta sección compara ambos métodos utilizando los **precios de mercado reales** obtenidos de IBKR.

**Metodología:**
- **BSM (Europeo):** Modelo Black-Scholes-Merton clásico (calculado arriba)
- **Americano (QuantLib):** Usa Barone-Adesi-Whaley para IV y Bjerksund-Stensland para griegas

**Nota:** Para opciones ATM con tenor corto (<30 días), las diferencias suelen ser mínimas.

In [9]:
# Calcular sigma promedio para el modelo americano
sigma_avg = (iv_call + iv_put) / 2

# Calcular griegas americanas usando precios de mercado reales
greeks_american = calculate_straddle_greeks_american(
    S, K, T, r, q, sigma_avg,
    call_price=call_mid,  # Precio de mercado (mid) del call
    put_price=put_mid     # Precio de mercado (mid) del put
)

# Mostrar comparación
print("="*70)
print("COMPARACIÓN: GRIEGAS BSM vs AMERICANAS (QuantLib)")
print("="*70)
print(f"\nParámetros utilizados:")
print(f"   S = ${S:.2f}, K = ${K:.0f}, T = {T:.4f} años")
print(f"   r = {r:.4f}, q = {q:.4f}, sigma = {sigma_avg:.4f}")
print(f"   Call price (market): ${call_mid:.2f}")
print(f"   Put price (market):  ${put_mid:.2f}")
print()
print(f"{'Griega':<10} {'BSM':>12} {'American':>12} {'Diferencia':>14} {'Diff %':>10}")
print("-"*70)

for g in ['delta', 'gamma', 'vega', 'theta', 'rho']:
    bsm_val = greeks[g]
    am_val = greeks_american[g]
    diff = am_val - bsm_val
    # Calcular diferencia porcentual (evitar división por cero)
    if abs(bsm_val) > 1e-8:
        diff_pct = (diff / abs(bsm_val)) * 100
        pct_str = f"{diff_pct:+.2f}%"
    else:
        pct_str = "N/A"
    print(f"{g.capitalize():<10} {bsm_val:>+12.4f} {am_val:>+12.4f} {diff:>+14.6f} {pct_str:>10}")

print("-"*70)
print(f"Método QuantLib: {greeks_american.get('method', 'N/A')}")
print("="*70)

COMPARACIÓN: GRIEGAS BSM vs AMERICANAS (QuantLib)

Parámetros utilizados:
   S = $695.16, K = $695, T = 0.0219 años
   r = 0.0370, q = -0.0083, sigma = 0.1049
   Call price (market): $4.75
   Put price (market):  $3.90

Griega              BSM     American     Diferencia     Diff %
----------------------------------------------------------------------
Delta           +0.0689      +0.0654      -0.003593     -5.21%
Gamma           +0.0736      +0.0749      +0.001286     +1.75%
Vega            +0.8182      +0.8175      -0.000715     -0.09%
Theta           -0.5417      -0.5414      +0.000369     +0.07%
Rho             +0.0086      +0.0140      +0.005406    +62.78%
----------------------------------------------------------------------
Método QuantLib: american_ql


## 6. Calculo del Hedge de Opciones

### Objetivo
Analizar diferentes estrategias de cobertura (hedge) para neutralizar el delta del straddle usando opciones adicionales.

### ¿Qué hace esta sección?

1. **Crea el objeto `StraddleForHedge`** con todos los datos del straddle calculados en el punto 5
2. **Busca 4 opciones de cobertura** en la cadena de IBKR:
   - Call ATM y Call OTM (2% fuera del dinero)
   - Put ATM y Put OTM (2% fuera del dinero)
3. **Analiza 8 estrategias de hedge**:
   - 4 estrategias de VENTA: Vender Call/Put ATM/OTM
   - 4 estrategias de COMPRA: Comprar Call/Put ATM/OTM
4. **Muestra tabla comparativa** con métricas clave:
   - Neutralidad delta (cercano a 0 es mejor)
   - Retención de gamma y vega (% del straddle original)
   - Cambio en theta
   - Prima pagada o cobrada

### Interpretación de resultados

| Estrategia | Delta | Gamma | Vega | Theta | Prima |
|------------|-------|-------|------|-------|-------|
| **Venta de opciones** | Neutraliza | Reduce | Reduce | Mejora (menos negativo) | Cobrada (+) |
| **Compra de opciones** | Neutraliza | Aumenta | Aumenta | Empeora (más negativo) | Pagada (-) |

### Variables utilizadas
- `S`, `K`, `T`, `r`, `q`: Parámetros del mercado (del punto 5)
- `greeks`: Griegas del straddle (del punto 5)
- `iv_call`, `iv_put`: Volatilidades implícitas (del punto 5)
- `expiry_date`: Fecha de vencimiento (del punto 2)

In [10]:
# Importar clases de hedge
from options_hedge import StraddleForHedge, OptionsHedgeAnalyzer

# Calcular sigma promedio para el hedge
sigma_avg = (iv_call + iv_put) / 2

# Crear objeto StraddleForHedge con los datos del punto 5
straddle = StraddleForHedge(
    spot=S,
    strike=K,
    expiry=expiry_date,
    T=T,
    r=r,
    q=q,
    sigma=sigma_avg,
    delta=greeks['delta'],
    gamma=greeks['gamma'],
    vega=greeks['vega'],
    theta=greeks['theta'],
    rho=greeks['rho']
)

print("="*60)
print("STRADDLE A CUBRIR")
print("="*60)
print(f"Spot:      ${straddle.spot:.2f}")
print(f"Strike:    ${straddle.strike:.0f}")
print(f"Expiry:    {straddle.expiry}")
print(f"T:         {straddle.T:.4f} años")
print(f"Sigma:     {straddle.sigma:.4f} ({straddle.sigma*100:.2f}%)")
print(f"\nGriegas del Straddle (por contrato):")
print(f"   Delta:  {straddle.delta:+.4f}")
print(f"   Gamma:  {straddle.gamma:.4f}")
print(f"   Vega:   {straddle.vega:.4f}")
print(f"   Theta:  {straddle.theta:.4f}")
print(f"   Rho:    {straddle.rho:.4f}")

STRADDLE A CUBRIR
Spot:      $695.16
Strike:    $695
Expiry:    20260121
T:         0.0219 años
Sigma:     0.1049 (10.49%)

Griegas del Straddle (por contrato):
   Delta:  +0.0689
   Gamma:  0.0736
   Vega:   0.8182
   Theta:  -0.5417
   Rho:    0.0086


In [11]:
# Crear analizador de hedge con df_option_chain para usar strikes consistentes
analyzer = OptionsHedgeAnalyzer(
    ib,
    otm_offset_pct=0.02,
    multiplier=100,
    df_option_chain=df_option_chain  # ← NUEVO: Usar cadena ya descargada
)

# Ejecutar análisis de las 4 estrategias apropiadas
results = analyzer.run_full_analysis(straddle, verbose=True)

ANALISIS DE ESTRATEGIAS DE COBERTURA

Straddle a cubrir:
   Spot: $695.16
   Strike: $695
   Expiry: 20260121
   Delta: +0.0689 (POSITIVA)
   Gamma: 0.0736
   Vega: 0.8182
   Theta: -0.5417

Buscando opciones de hedge...
ATM Target Strike: 695
   -> Buscando CALL ATM (K~$695)... K=$695 (desde DataFrame)
K=$695, Precio=$4.75
OTM Call Target Strike: 709
   -> Buscando CALL OTM (K~$709)... K=$709 (desde DataFrame)
K=$709, Precio=$0.23
ATM Target Strike: 695
   -> Buscando PUT ATM (K~$695)... K=$695 (desde DataFrame)
K=$695, Precio=$3.90
OTM Put Target Strike: 681
   -> Buscando PUT OTM (K~$681)... K=$681 (desde DataFrame)
K=$681, Precio=$0.96

Delta POSITIVA detectada → Estrategias: Venta Calls + Compra Puts
Analizando 4 estrategias de hedge apropiadas...

----------------------------------------------------------------------
1. Venta Call ATM
----------------------------------------------------------------------
Accion: VENDER 0.13 contratos del CALL K=$695
   (Delta hedge: +0.5346, impa

In [12]:
# Crear tabla comparativa de todas las estrategias
import pandas as pd

# Construir datos para el DataFrame
comparison_data = []
for a in results:
    comparison_data.append({
        'Estrategia': a.strategy_name,
        'Contratos': f"{a.hedge.contracts:+.2f}",
        'Delta Final': f"{a.total_delta:.4f}",
        'Gamma %': f"{a.gamma_retention_pct:.1f}%",
        'Vega %': f"{a.vega_retention_pct:.1f}%",
        'Theta %': f"{a.theta_change_pct:+.1f}%",
        'Prima ($)': f"${a.hedge_premium:+,.2f}"
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("TABLA COMPARATIVA DE ESTRATEGIAS DE HEDGE")
print("="*100)
print("\nInterpretacion:")
print("  - Delta Final: Cercano a 0 = mejor neutralizacion")
print("  - Gamma/Vega %: Porcentaje retenido del straddle original (>100% = aumenta)")
print("  - Theta %: Cambio vs straddle original (+ = mejora, - = empeora)")
print("  - Prima: + = cobramos, - = pagamos")
print("\n")
print(df_comparison.to_string(index=False))
print("\n" + "="*100)


TABLA COMPARATIVA DE ESTRATEGIAS DE HEDGE

Interpretacion:
  - Delta Final: Cercano a 0 = mejor neutralizacion
  - Gamma/Vega %: Porcentaje retenido del straddle original (>100% = aumenta)
  - Theta %: Cambio vs straddle original (+ = mejora, - = empeora)
  - Prima: + = cobramos, - = pagamos


       Estrategia Contratos Delta Final Gamma % Vega % Theta % Prima ($)
1. Venta Call ATM     -0.13     -0.0000   93.6%  93.6%   +7.5%   $-61.27
2. Venta Call OTM     -0.60      0.0000   85.4%  85.4%  +15.6%   $-13.71
3. Compra Put ATM     +0.15      0.0000  107.4% 107.4%   -6.2%   $+57.75
4. Compra Put OTM     +0.85      0.0000  116.1% 116.1%  -14.8%   $+81.40



### 6.1 Comparación de Hedge: Griegas BSM vs Americanas

Esta sección analiza cómo cambian los resultados del hedge si utilizamos las griegas calculadas con el modelo americano (QuantLib) en lugar de BSM.

**Objetivo:** Cuantificar el impacto de usar diferentes modelos de pricing en las decisiones de cobertura.

In [13]:
# Crear straddle con griegas americanas para comparación de hedge
straddle_american = StraddleForHedge(
    spot=S,
    strike=float(K),
    expiry=expiry_date,
    T=T,
    r=r,
    q=q,
    sigma=sigma_avg,
    delta=greeks_american['delta'],
    gamma=greeks_american['gamma'],
    vega=greeks_american['vega'],
    theta=greeks_american['theta'],
    rho=greeks_american['rho']
)


In [14]:
type(r)

numpy.float64

In [15]:

print("="*80)
print("STRADDLE CON GRIEGAS AMERICANAS")
print("="*80)
print(f"Delta:  {straddle_american.delta:+.4f} (BSM: {straddle.delta:+.4f})")
print(f"Gamma:  {straddle_american.gamma:.4f} (BSM: {straddle.gamma:.4f})")
print(f"Vega:   {straddle_american.vega:.4f} (BSM: {straddle.vega:.4f})")
print(f"Theta:  {straddle_american.theta:.4f} (BSM: {straddle.theta:.4f})")

# Ejecutar análisis de hedge con griegas americanas (sin verbose)
print("\nEjecutando análisis de hedge con griegas americanas...")
results_american = analyzer.run_full_analysis(straddle_american, verbose=False)

STRADDLE CON GRIEGAS AMERICANAS
Delta:  +0.0654 (BSM: +0.0689)
Gamma:  0.0749 (BSM: 0.0736)
Vega:   0.8175 (BSM: 0.8182)
Theta:  -0.5414 (BSM: -0.5417)

Ejecutando análisis de hedge con griegas americanas...


In [16]:
# Crear tabla comparativa BSM vs American para hedge
comparison_data_combined = []

for r_bsm, r_am in zip(results, results_american):
    comparison_data_combined.append({
        'Estrategia': r_bsm.strategy_name,
        'Contratos BSM': f"{r_bsm.hedge.contracts:+.2f}",
        'Contratos AM': f"{r_am.hedge.contracts:+.2f}",
        'Delta BSM': f"{r_bsm.total_delta:.4f}",
        'Delta AM': f"{r_am.total_delta:.4f}",
        'Gamma% BSM': f"{r_bsm.gamma_retention_pct:.1f}%",
        'Gamma% AM': f"{r_am.gamma_retention_pct:.1f}%",
        'Prima BSM': f"${r_bsm.hedge_premium:+,.2f}",
        'Prima AM': f"${r_am.hedge_premium:+,.2f}"
    })

df_comparison_combined = pd.DataFrame(comparison_data_combined)

print("\n" + "="*120)
print("TABLA COMPARATIVA DE HEDGE: BSM vs AMERICAN")
print("="*120)
print("\nInterpretación:")
print("  - Contratos: Número de contratos necesarios para neutralizar delta")
print("  - Delta: Debe ser ~0 para cobertura efectiva")
print("  - Gamma%: Porcentaje de gamma retenido del straddle original")
print("  - Prima: Costo (+pagada) o ingreso (-cobrada) del hedge")
print("\n")
print(df_comparison_combined.to_string(index=False))

# Calcular diferencia en contratos
print("\n" + "-"*120)
print("DIFERENCIA EN NÚMERO DE CONTRATOS (American - BSM):")
print("-"*120)
for r_bsm, r_am in zip(results, results_american):
    diff_contracts = r_am.hedge.contracts - r_bsm.hedge.contracts
    diff_pct = (diff_contracts / abs(r_bsm.hedge.contracts)) * 100 if r_bsm.hedge.contracts != 0 else 0
    print(f"  {r_bsm.strategy_name}: {diff_contracts:+.4f} contratos ({diff_pct:+.2f}%)")

print("\n" + "="*120)


TABLA COMPARATIVA DE HEDGE: BSM vs AMERICAN

Interpretación:
  - Contratos: Número de contratos necesarios para neutralizar delta
  - Delta: Debe ser ~0 para cobertura efectiva
  - Gamma%: Porcentaje de gamma retenido del straddle original
  - Prima: Costo (+pagada) o ingreso (-cobrada) del hedge


       Estrategia Contratos BSM Contratos AM Delta BSM Delta AM Gamma% BSM Gamma% AM Prima BSM Prima AM
1. Venta Call ATM         -0.13        -0.12   -0.0000   0.0000      93.6%     94.0%   $-61.27  $-58.07
2. Venta Call OTM         -0.60        -0.57    0.0000   0.0000      85.4%     86.4%   $-13.71  $-13.00
3. Compra Put ATM         +0.15        +0.14    0.0000   0.0000     107.4%    106.9%   $+57.75  $+54.74
4. Compra Put OTM         +0.85        +0.80    0.0000   0.0000     116.1%    115.0%   $+81.40  $+77.15

------------------------------------------------------------------------------------------------------------------------
DIFERENCIA EN NÚMERO DE CONTRATOS (American - BSM):
-----

### Analisis de Resultados

**Estrategias de VENTA (prima cobrada):**
- Reducen gamma y vega del portafolio
- Mejoran theta (menos decay)
- Ideales si esperas baja volatilidad realizada

**Estrategias de COMPRA (prima pagada):**
- Aumentan gamma y vega del portafolio  
- Empeoran theta (mas decay)
- Ideales si esperas alta volatilidad realizada

**Recomendacion general:**
- Para cobertura delta pura con minimo costo: opciones OTM
- Para mantener exposicion a volatilidad: opciones ATM
- Venta vs Compra depende de tu vision de volatilidad futura

---

## [LAST] Desconectar de TWS/IB Gateway

### Objetivo
Cerrar la conexión con IBKR de forma limpia y liberar el CLIENT_ID.

### ¿Qué hace esta celda?

#### **Función `disconnect_from_ib_async()`**

1. **Verifica si hay conexión activa**
   - Comprueba `ib.isConnected()`
   - Si no hay conexión, muestra advertencia y termina

2. **Cancela suscripciones activas**
   - Cancela todas las suscripciones de market data
   - Limpia recursos antes de desconectar
   - Previene warnings de IBKR

3. **Desconecta limpiamente**
   - Ejecuta `ib.disconnect()`
   - Espera 1 segundo para asegurar desconexión completa
   - Libera el CLIENT_ID para futuros usos

4. **Confirma desconexión**
   - Muestra CLIENT_ID liberado
   - Confirma que la sesión finalizó correctamente

### Resultado esperado



In [17]:
async def disconnect_from_ib_async():
    """Desconecta de TWS/IB Gateway de forma segura"""
    
    print("="*80)
    print("CERRANDO CONEXIÓN")
    print("="*80 + "\n")
    
    try:
        if ib.isConnected():
            client_id = ib.client.clientId
            print(f"Desconectando clientId={client_id}...")
            
            # Cancelar todas las suscripciones de market data activas
            try:
                ib.cancelMktData()
            except:
                pass
            
            # Desconectar
            ib.disconnect()
            await asyncio.sleep(1)
            
            print("✓ Desconectado exitosamente de TWS/IB Gateway")
            print(f"  ClientId {client_id} liberado\n")
        else:
            print("⚠ No hay conexión activa para cerrar\n")
            
    except Exception as e:
        print(f"✗ Error al desconectar: {e}")
        print("  (La conexión puede haberse cerrado ya)\n")

# Ejecutar desconexión
await disconnect_from_ib_async()
print("✓ Sesión finalizada")

CERRANDO CONEXIÓN

Desconectando clientId=178...
⚠ Desconectado de TWS/IB Gateway
✓ Desconectado exitosamente de TWS/IB Gateway
  ClientId 178 liberado

✓ Sesión finalizada
